# Route Origin Validation on Real RPKI Data

In this exercise you will take a set of real IP addresses, look them up against a real
snapshot of validated RPKI data, and classify each one as **Valid**, **Invalid**, or
**Not Found**.

---
## 1. The dataset: historical validated ROA payloads

RIPE NCC publishes a **daily historical archive** of validated ROA payloads. Despite being
hosted by RIPE NCC, these archives contain the validated output from **all five** trust
anchors, not only RIPE NCC's own resources. Each daily file is a CSV listing one row per
VRP, which makes it possible to reconstruct the exact RPKI picture as it stood on any past
day and ask what a validating router would have concluded at that moment.

One file is published per trust anchor per day:

```
https://ftp.ripe.net/ripe/rpki/<ta>.tal/<YYYY>/<MM>/<DD>/roas.csv.xz

<ta> is one of:  afrinic, ripencc, arin, lacnic, apnic
```

Note the identifier for the RIPE NCC trust anchor is **`ripencc`**, not `ripe`. The loader
below uses that spelling, so the values you will see in the `trust_anchor` column are
`arin`, `ripencc`, `apnic`, `lacnic`, and `afrinic`.

After downloading, parsing, and normalizing the RIPE CSV columns, `load_rpki_snapshot()`
returns a DataFrame with **one row per VRP**:

| field | example | description |
| --- | --- | --- |
| `prefix` | `137.110.0.0/16` | prefix authorized by the ROA (normalized) |
| `max_length` | `20` | maximum prefix length allowed |
| `asn` | `7377` | AS authorized to originate the prefix |
| `trust_anchor` | `arin` | RPKI trust anchor that validated the VRP |
| `family` | `4` | IP address family (4 or 6) |
| `date` | `2023-03-01` | snapshot date the rows were loaded for |

Example rows:

```
prefix,max_length,asn,trust_anchor,family
137.110.0.0/16,20,7377,arin,4
35.228.224.0/19,19,15169,ripencc,4
192.0.64.0/18,18,2635,arin,4
```

> **Helpful hint:** the `trust_anchor` field identifies the RPKI trust anchor that validated
> the ROA payload. It is useful for grouping VRPs by RIR, but RPKI validation itself depends
> only on **prefix coverage**, **origin AS**, and **maximum length** — never on which trust
> anchor a row came from.

In [5]:
%pip install pandas
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import ipaddress
import lzma
import io
from pathlib import Path

import pandas as pd
import requests

# The five RPKI trust anchors, one per RIR.
TRUST_ANCHORS = ["afrinic", "ripencc", "arin", "lacnic", "apnic"]

# The snapshot day to analyze. Any day the archive covers will work.
SNAPSHOT_DATE = "2023-03-01"

ARCHIVE_URL = "https://ftp.ripe.net/ripe/rpki/{ta}.tal/{yyyy}/{mm}/{dd}/roas.csv.xz"

# Downloaded files are cached here so re-running the notebook doesn't re-fetch them.
CACHE_DIR = Path("rpki_cache")
CACHE_DIR.mkdir(exist_ok=True)

print(f"Trust anchors: {', '.join(TRUST_ANCHORS)}")
print(f"Snapshot date: {SNAPSHOT_DATE}")

Trust anchors: afrinic, ripencc, arin, lacnic, apnic
Snapshot date: 2023-03-01


### The loader (provided)

`load_rpki_snapshot()` downloads one `roas.csv.xz` per trust anchor for the given day,
decompresses it, normalizes the RIPE column names and value formats (for example stripping
the `AS` prefix off ASNs and filling in a missing max length with the prefix's own length),
and concatenates everything into a single DataFrame.

In [7]:
def _fetch_vrp_csv(trust_anchor: str, date: str) -> bytes:
    """Download (or read from cache) one trust anchor's VRP file for one day."""
    yyyy, mm, dd = date.split("-")
    cache_path = CACHE_DIR / f"{trust_anchor}-{date}.csv"

    if cache_path.exists():
        return cache_path.read_bytes()

    url = ARCHIVE_URL.format(ta=trust_anchor, yyyy=yyyy, mm=mm, dd=dd)
    response = requests.get(url, timeout=180)
    response.raise_for_status()

    raw = lzma.decompress(response.content)
    cache_path.write_bytes(raw)
    return raw


def _normalize(df: pd.DataFrame, trust_anchor: str, date: str) -> pd.DataFrame:
    """Map RIPE's CSV columns onto the schema described above."""
    # RIPE's header has varied over the years; match it case-insensitively.
    lookup = {c.strip().lower(): c for c in df.columns}
    asn_col = lookup.get("asn")
    prefix_col = lookup.get("ip prefix", lookup.get("prefix"))
    maxlen_col = lookup.get("max length", lookup.get("maxlength"))

    out = pd.DataFrame()
    out["prefix"] = df[prefix_col].astype(str).str.strip()

    # ASNs appear as "AS7377" or "7377" depending on the vintage of the file.
    out["asn"] = (
        df[asn_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"^AS", "", regex=True)
        .astype(int)
    )

    # A blank max length means "exactly the prefix's own length".
    prefix_len = out["prefix"].str.split("/").str[1].astype(int)
    out["max_length"] = (
        pd.to_numeric(df[maxlen_col], errors="coerce").fillna(prefix_len).astype(int)
    )

    out["trust_anchor"] = trust_anchor
    out["family"] = out["prefix"].str.contains(":").map({True: 6, False: 4})
    out["date"] = date

    return out[["prefix", "max_length", "asn", "trust_anchor", "family", "date"]]


def load_rpki_snapshot(date: str = SNAPSHOT_DATE, trust_anchors=TRUST_ANCHORS) -> pd.DataFrame:
    """Return one row per VRP across all trust anchors for the given day."""
    frames = []
    for ta in trust_anchors:
        raw = _fetch_vrp_csv(ta, date)
        df = pd.read_csv(io.BytesIO(raw))
        frames.append(_normalize(df, ta, date))
        print(f"  {ta:<8} {len(frames[-1]):>7,} VRPs")

    return pd.concat(frames, ignore_index=True)

In [8]:
print(f"Loading VRPs for {SNAPSHOT_DATE} ...")
vrps = load_rpki_snapshot()

print(f"\nTotal: {len(vrps):,} VRPs")
vrps.head()

Loading VRPs for 2023-03-01 ...
  afrinic    4,583 VRPs
  ripencc  191,012 VRPs
  arin      72,620 VRPs
  lacnic    31,616 VRPs
  apnic    104,845 VRPs

Total: 404,676 VRPs


,prefix,max_length,asn,trust_anchor,family,date
0,41.0.0.0/16,24,36994,afrinic,4,2023-03-01
1,41.10.0.0/16,24,36994,afrinic,4,2023-03-01
2,41.21.128.0/17,24,36994,afrinic,4,2023-03-01
3,41.23.0.0/16,24,36994,afrinic,4,2023-03-01
4,41.61.0.0/16,24,36943,afrinic,4,2023-03-01


In [9]:
# A quick look at how the snapshot breaks down.
print("VRPs per trust anchor:")
print(vrps["trust_anchor"].value_counts().to_string())

print("\nVRPs per address family:")
print(vrps["family"].value_counts().to_string())

VRPs per trust anchor:
trust_anchor
ripencc    191012
apnic      104845
arin        72620
lacnic      31616
afrinic      4583

VRPs per address family:
family
4    333163
6     71513


---
## 2. ROV as a table lookup

With VRPs in hand, classifying a route reduces to a three-step lookup. For an address or
prefix **P** announced by origin **AS A**:

```
  1. Find every VRP whose prefix COVERS P.
       none found                                       -> NOT FOUND
  2. Among those covering VRPs, is there at least one where
       origin ASN == A  AND  P's length <= max_length?
       yes                                              -> VALID
       no  (covered, but nothing authorizes A at this
            length)                                     -> INVALID
```

Two subtleties are worth holding onto.

First, the difference between the two negative states: **Not Found** means *no ROA covers
this prefix at all*, while **Invalid** means *a ROA covers it and affirmatively does not
authorize this announcement*. These are very different signals to an operator, and mixing
them up is the most common bug in a first ROV implementation.

Second, a single covering VRP that authorizes the route is enough to make it **Valid** — so
every covering VRP must be checked before you may conclude Invalid. A prefix can
legitimately be covered by several ROAs at once, with different origins, different max
lengths, or even different trust anchors.

### The addresses to classify

Each target below is a real IP address, together with the prefix it was announced in and the
AS that originated that announcement — the two facts a router would have taken from the BGP
update carrying the route.

Some of these announcements are genuine and some are not. Several addresses appear more than
once, announced at different prefix lengths or by different origins.

In [ ]:
# (ip_address, announced_prefix, origin_asn)
TARGETS = [
    ("8.8.8.8", "8.8.8.0/24", 15169),
    ("8.8.8.8", "8.8.8.0/24", 4134),
    ("1.1.1.1", "1.1.1.0/24", 13335),
    ("1.1.1.1", "1.1.1.0/24", 7007),
    ("192.0.66.20", "192.0.64.0/18", 2635),
    ("192.0.66.20", "192.0.66.0/24", 2635),
    ("192.0.66.200", "192.0.66.128/25", 2635),
    ("208.65.153.238", "208.65.152.0/22", 36040),
    ("208.65.153.238", "208.65.153.0/24", 17557),
    ("137.110.222.10", "137.110.0.0/16", 7377),
    ("198.51.100.7", "198.51.100.0/24", 64511),
]

print(f"{len(TARGETS)} announcements to classify")

---
## 3. Your task: implement ROV

Fill in `classify_rov()` below so that it implements the three-step lookup above and returns
one of the strings `"Valid"`, `"Invalid"`, or `"Not Found"`.

Python's standard-library [`ipaddress`](https://docs.python.org/3/library/ipaddress.html)
module is already imported and does the hard part of coverage testing for you:

```python
net = ipaddress.ip_network("137.110.222.0/24")
roa = ipaddress.ip_network("137.110.0.0/16")

net.subnet_of(roa)   # True  -> the ROA prefix covers this announcement
net.prefixlen        # 24    -> compare this against the VRP's max_length
```

One trap to watch for: `subnet_of()` raises a `TypeError` when the two networks belong to
different address families, and the snapshot holds both IPv4 and IPv6 VRPs. Compare
`.version` first, or filter the table by its `family` column.

Don't worry about speed on your first pass — a plain scan over the rows is fine. Getting the
three states right matters far more than getting there quickly.

In [ ]:
def classify_rov(announced_prefix: str, origin_asn: int, vrps: pd.DataFrame) -> str:
    """Classify a BGP announcement against a table of VRPs.

    Args:
        announced_prefix: the prefix as announced in BGP, e.g. "137.110.0.0/16"
        origin_asn:       the AS that originated the announcement, e.g. 7377
        vrps:             DataFrame with columns prefix, max_length, asn, ...

    Returns:
        One of "Valid", "Invalid", or "Not Found".
    """
    # TODO: implement the three-step lookup.
    #
    #   1. Collect every VRP whose prefix covers `announced_prefix`.
    #      If there are none, the announcement is "Not Found".
    #
    #   2. Among the covering VRPs, look for one that authorizes this announcement:
    #      its asn matches `origin_asn` AND the announced prefix length is
    #      no longer than its max_length.  If you find one, it is "Valid".
    #
    #   3. Covered, but nothing authorized it -> "Invalid".

    raise NotImplementedError("classify_rov is yours to write")

### Check yourself on a hand-built table

Before running against 400,000 real VRPs, test your function on a tiny table you can reason
about by hand. Every case below should pass; each one isolates a different part of the rule.

In [ ]:
TEST_VRPS = pd.DataFrame(
    [
        # prefix,           max_length, asn,   trust_anchor, family, date
        ("203.0.113.0/24", 24, 64500, "arin", 4, "test"),
        ("198.51.100.0/24", 26, 64501, "ripencc", 4, "test"),
        ("192.0.2.0/24", 24, 64502, "apnic", 4, "test"),
        ("192.0.2.0/24", 24, 64503, "lacnic", 4, "test"),  # same prefix, second origin
        ("2001:db8::/32", 48, 64504, "ripencc", 6, "test"),
    ],
    columns=["prefix", "max_length", "asn", "trust_anchor", "family", "date"],
)

TEST_CASES = [
    # (prefix, origin, expected, what it exercises)
    ("203.0.113.0/24", 64500, "Valid", "exact match on prefix and origin"),
    ("203.0.113.0/24", 66000, "Invalid", "covered, wrong origin"),
    ("203.0.113.128/25", 64500, "Invalid", "right origin, beyond max length"),
    ("203.0.113.128/25", 66000, "Invalid", "sub-prefix hijack"),
    ("198.51.100.192/26", 64501, "Valid", "more specific, but within max length"),
    ("198.51.100.192/28", 64501, "Invalid", "more specific than max length allows"),
    ("192.0.2.0/24", 64503, "Valid", "second covering VRP authorizes it"),
    ("2001:db8:1::/48", 64504, "Valid", "IPv6, within max length"),
    ("10.0.0.0/8", 64500, "Not Found", "no VRP covers this prefix at all"),
]

passed = 0
for prefix, origin, expected, description in TEST_CASES:
    actual = classify_rov(prefix, origin, TEST_VRPS)
    ok = actual == expected
    passed += ok
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {prefix:<18} AS{origin:<6} got {actual:<9} "
          f"expected {expected:<9} ({description})")

print(f"\n{passed}/{len(TEST_CASES)} cases passed")

---
## 4. Classify the real addresses

Once the hand-built cases all pass, run your function against the real snapshot.

In [ ]:
results = pd.DataFrame(
    [
        {
            "ip": ip,
            "announced_prefix": prefix,
            "origin_asn": origin,
            "rov_state": classify_rov(prefix, origin, vrps),
        }
        for ip, prefix, origin in TARGETS
    ]
)

results

In [ ]:
print("How the announcements broke down:")
print(results["rov_state"].value_counts().to_string())

---
## 5. Reflect

Answer these using your results above.

1. Three of the targets announce the same address space from the same origin AS at three
   different prefix lengths (`/18`, `/24`, `/25`). Did all three get the same state? Explain
   the boundary in terms of the covering ROA's max length.
2. Two targets came back **Not Found**. What would a router configured with standard policy
   do with them, and why is accepting them still the safe choice today?
3. Pick an announcement that came back **Invalid**. Print the covering VRPs for its prefix
   and explain exactly which condition it failed — wrong origin, excessive length, or both.
4. Your function ignores the `trust_anchor` column entirely. Convince yourself this is
   correct: what would go wrong if you had filtered the VRPs by trust anchor first?

In [ ]:
# Scratch space for the reflection questions.
#
# Hint for Q3 - listing the VRPs that cover a prefix:
#
#   net = ipaddress.ip_network("...")
#   covering = vrps[vrps["prefix"].apply(
#       lambda p: ipaddress.ip_network(p).version == net.version
#                 and net.subnet_of(ipaddress.ip_network(p))
#   )]
#   covering
